In [7]:
# ─── CELL 0: ENVIRONMENT SETUP ───────────────────────────────────────────────
import os

if os.path.exists('/kaggle/working'):
    ENV         = 'kaggle'
    BASE_PATH   = '/kaggle/input/datasets/ascanipek/eyepacs-aptos-messidor-diabetic-retinopathy'
    CKPT_DIR    = '/kaggle/working'
    NUM_WORKERS = 4
else:
    ENV         = 'colab'
    import kagglehub
    BASE_PATH   = kagglehub.dataset_download('ascanipek/eyepacs-aptos-messidor-diabetic-retinopathy')
    CKPT_DIR    = '/content'
    NUM_WORKERS = 2

print(f'Environment : {ENV}')
print(f'Base path   : {BASE_PATH}')
print(f'Checkpoint  : {CKPT_DIR}')
print(f'Workers     : {NUM_WORKERS}')


Environment : kaggle
Base path   : /kaggle/input/datasets/ascanipek/eyepacs-aptos-messidor-diabetic-retinopathy
Checkpoint  : /kaggle/working
Workers     : 4


In [2]:
# ─── CELL 1: IMPORTS & SETUP ─────────────────────────────────────────────────
!pip install -q timm albumentations

import os
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import cohen_kappa_score, accuracy_score, classification_report, roc_auc_score
from sklearn.preprocessing import label_binarize
import albumentations as A
from albumentations.pytorch import ToTensorV2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import timm
from tqdm.notebook import tqdm

def seed_everything(seed=42):
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

seed_everything()
print('Setup Complete.')


Setup Complete.


In [15]:
# ─── CELL 2: CONFIGURATION ───────────────────────────────────────────────────
class Config:
    seed        = 42
    img_size    = 224
    num_classes = 5
    batch_size  = 16
    epochs      = 20
    n_folds     = 5
    device      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model_name  = 'swin'  # change to 'swin' or 'maxvit' for other models

    base_path     = BASE_PATH
    train_dir     = os.path.join(BASE_PATH, 'augmented_resized_V2', 'train')
    csv_save_path = os.path.join(CKPT_DIR, 'dataset_map.csv')

print(f'Configuration Loaded.')
print(f'Device    : {Config.device}')
print(f'Train dir : {Config.train_dir}')
print(f'CSV path  : {Config.csv_save_path}')


Configuration Loaded.
Device    : cuda
Train dir : /kaggle/input/datasets/ascanipek/eyepacs-aptos-messidor-diabetic-retinopathy/augmented_resized_V2/train
CSV path  : /kaggle/working/dataset_map.csv


In [9]:
# ─── CELL 3: DATA MAPPING ────────────────────────────────────────────────────
def create_dataset_map():
    if os.path.exists(Config.csv_save_path):
        print(f'Loading existing map from {Config.csv_save_path}')
        return pd.read_csv(Config.csv_save_path)

    print('Scanning folders to build dataset map...')
    data = []

    try:
        classes = sorted([d for d in os.listdir(Config.train_dir)
                          if os.path.isdir(os.path.join(Config.train_dir, d))])
    except FileNotFoundError:
        print('ERROR: Training folder not found. Check Config path.')
        return pd.DataFrame()

    print(f'Found classes: {classes}')

    for class_name in classes:
        class_path = os.path.join(Config.train_dir, class_name)
        try:
            label = int(class_name)
        except ValueError:
            mapping = {'No_DR': 0, 'Mild': 1, 'Moderate': 2, 'Severe': 3, 'Proliferate_DR': 4}
            label = mapping.get(class_name, -1)
        if label == -1: continue
        files = [f for f in os.listdir(class_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        for f in files:
            data.append({'image_path': os.path.join(class_path, f), 'label': label})

    df = pd.DataFrame(data)
    skf = StratifiedKFold(n_splits=Config.n_folds, shuffle=True, random_state=Config.seed)
    df['fold'] = -1
    for fold, (_, val_idx) in enumerate(skf.split(df, df['label'])):
        df.loc[val_idx, 'fold'] = fold

    df.to_csv(Config.csv_save_path, index=False)
    print(f'Map created with {len(df)} images.')
    return df

df = create_dataset_map()
print(df.head())
print('\nClass distribution:')
print(df['label'].value_counts().sort_index())


Scanning folders to build dataset map...
Found classes: ['0', '1', '2', '3', '4']
Map created with 115241 images.
                                          image_path  label  fold
0  /kaggle/input/datasets/ascanipek/eyepacs-aptos...      0     1
1  /kaggle/input/datasets/ascanipek/eyepacs-aptos...      0     0
2  /kaggle/input/datasets/ascanipek/eyepacs-aptos...      0     3
3  /kaggle/input/datasets/ascanipek/eyepacs-aptos...      0     4
4  /kaggle/input/datasets/ascanipek/eyepacs-aptos...      0     0

Class distribution:
label
0    55162
1    18470
2    24198
3     7936
4     9475
Name: count, dtype: int64


In [11]:
# ─── CELL 4: DATASET & TRANSFORMS ────────────────────────────────────────────

def circular_crop(img):
    """Remove black border around retinal image."""
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    mask = gray > 7
    if mask.any():
        row_idx = np.where(np.any(mask, axis=1))[0]
        col_idx = np.where(np.any(mask, axis=0))[0]
        r0, r1 = int(row_idx[0]),  int(row_idx[-1])
        c0, c1 = int(col_idx[0]),  int(col_idx[-1])
        if r1 > r0 and c1 > c0 and (r1 - r0) >= 8 and (c1 - c0) >= 8:
            img = img[r0:r1+1, c0:c1+1]
    return img


class DRDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df        = df
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        img_path = row['image_path']
        label    = row['label']

        image = cv2.imread(img_path)
        if image is None:
            image = np.zeros((Config.img_size, Config.img_size, 3), dtype=np.uint8)
        else:
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            image = circular_crop(image)  # remove black border

        if self.transform:
            augmented = self.transform(image=image)
            image     = augmented['image']

        return image, torch.tensor(label, dtype=torch.long)


def get_transforms(data='train'):
    if data == 'train':
        return A.Compose([
            A.Resize(Config.img_size, Config.img_size),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.RandomRotate90(p=0.5),
            A.RandomBrightnessContrast(p=0.4),
            A.HueSaturationValue(p=0.3),
            A.GaussNoise(p=0.3),
            A.CLAHE(clip_limit=4.0, p=0.7),
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2(),
        ])
    else:
        return A.Compose([
            A.Resize(Config.img_size, Config.img_size),
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2(),
        ])


print('Dataset pipeline ready.')


Dataset pipeline ready.


In [12]:
# ─── CELL 5: MODEL ARCHITECTURE ──────────────────────────────────────────────
class DRModel(nn.Module):
    def __init__(self, model_name, num_classes=5, pretrained=True):
        super(DRModel, self).__init__()

        if model_name == 'convnext':
            self.backbone = timm.create_model('convnext_tiny', pretrained=pretrained, num_classes=0)
        elif model_name == 'swin':
            self.backbone = timm.create_model('swin_base_patch4_window7_224', pretrained=pretrained, num_classes=0)
        elif model_name == 'maxvit':
            self.backbone = timm.create_model('maxvit_tiny_tf_224', pretrained=pretrained, num_classes=0)
        elif model_name == 'effnet':
            self.backbone = timm.create_model('tf_efficientnet_b4', pretrained=pretrained, num_classes=0)

        self.n_features = self.backbone.num_features
        self.drop       = nn.Dropout(p=0.3)
        self.fc         = nn.Linear(self.n_features, num_classes)

    def forward(self, x):
        features = self.backbone(x)
        x        = self.drop(features)
        return self.fc(x)


print('Model architecture defined.')
# Quick param count
for name in ['convnext', 'swin']:
    m = DRModel(name, pretrained=False)
    params = sum(p.numel() for p in m.parameters()) / 1e6
    print(f'  {name:<12}: {params:.1f}M parameters')
    del m


Model architecture defined.
  convnext    : 27.8M parameters
  swin        : 86.7M parameters


In [13]:
# ─── CELL 6: LOSS & UTILS ────────────────────────────────────────────────────
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha     = alpha
        self.gamma     = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        CE_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt      = torch.exp(-CE_loss)

        if self.alpha is None:
            alpha_t = 1.0
        elif isinstance(self.alpha, torch.Tensor):
            alpha_t = self.alpha.to(inputs.device)[targets]
        else:
            alpha_t = self.alpha

        F_loss = alpha_t * (1 - pt) ** self.gamma * CE_loss
        if self.reduction == 'mean':
            return torch.mean(F_loss)
        return F_loss


def compute_class_alpha(df, num_classes, device):
    counts = df['label'].value_counts().sort_index().values.astype(float)
    weights = 1.0 / counts
    alpha = torch.tensor(weights / weights.sum(), dtype=torch.float).to(device)
    print(f'Class alpha weights: {alpha.cpu().numpy().round(4)}')
    return alpha


def make_weighted_sampler(train_df):
    counts = train_df['label'].value_counts().sort_index().values.astype(float)
    sample_weights = [1.0 / counts[label] for label in train_df['label']]
    return WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(sample_weights),
        replacement=True
    )


def train_one_epoch(model, loader, criterion, optimizer, scaler, device):
    model.train()
    running_loss = 0.0
    pbar = tqdm(loader, desc='Training')
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            outputs = model(images)
            loss    = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item()
        pbar.set_postfix(loss=running_loss / len(loader))
    return running_loss / len(loader)


@torch.no_grad()
def valid_one_epoch(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    preds_all, labels_all = [], []
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss    = criterion(outputs, labels)
        running_loss += loss.item()
        preds_all.append(outputs.softmax(1).cpu().numpy())
        labels_all.append(labels.cpu().numpy())
    preds_all  = np.concatenate(preds_all)
    labels_all = np.concatenate(labels_all)
    predictions = np.argmax(preds_all, axis=1)
    qwk = cohen_kappa_score(labels_all, predictions, weights='quadratic')
    acc = accuracy_score(labels_all, predictions)
    return running_loss / len(loader), qwk, acc


print('Training utilities ready.')


Training utilities ready.


In [ ]:
# ─── CELL 7: TRAINING (All 5 Folds) ──────────────────────────────────────────
# Set Config.model_name before running:
#   'convnext' → ConvNeXt-Tiny  (28M)
#   'swin'     → Swin-Base      (88M)

print(f'[INFO] Checkpoint directory: {CKPT_DIR}')
print(f'[INFO] Training model: {Config.model_name}')


def run_training(start_fold=0, resume_ckpt_dir=None):
    """
    start_fold:       which fold to start from (0 = all folds)
    resume_ckpt_dir:  directory with existing checkpoints to resume from
                      if None, starts from ImageNet pretrained weights
    """
    df    = pd.read_csv(Config.csv_save_path)
    alpha = compute_class_alpha(df, Config.num_classes, Config.device)

    print(f'\n[INFO] Training folds {start_fold} to {Config.n_folds - 1}')

    for fold in range(start_fold, Config.n_folds):
        print(f"\n{'='*50}")
        print(f'  FOLD {fold} / {Config.n_folds - 1}')
        print(f"{'='*50}")

        ckpt_path = os.path.join(CKPT_DIR, f'{Config.model_name}_fold{fold}_best.pth')

        train_df = df[df['fold'] != fold].reset_index(drop=True)
        valid_df = df[df['fold'] == fold].reset_index(drop=True)

        train_ds = DRDataset(train_df, transform=get_transforms('train'))
        valid_ds = DRDataset(valid_df, transform=get_transforms('valid'))

        sampler = make_weighted_sampler(train_df)
        train_loader = DataLoader(
            train_ds, batch_size=Config.batch_size,
            sampler=sampler, num_workers=NUM_WORKERS, pin_memory=True
        )
        valid_loader = DataLoader(
            valid_ds, batch_size=Config.batch_size,
            shuffle=False, num_workers=NUM_WORKERS, pin_memory=True
        )

        # Load weights
        model = DRModel(Config.model_name, num_classes=Config.num_classes).to(Config.device)
        if resume_ckpt_dir is not None:
            resume_path = os.path.join(resume_ckpt_dir, f'{Config.model_name}_fold{fold}_best.pth')
            if os.path.exists(resume_path):
                model.load_state_dict(torch.load(resume_path, map_location=Config.device))
                print(f'[RESUME] Loaded checkpoint: {resume_path}')
            else:
                print(f'[FRESH] No checkpoint found, starting from ImageNet weights')
        else:
            print(f'[FRESH] Starting from ImageNet weights')

        optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
        criterion = FocalLoss(alpha=alpha, gamma=2.0)
        scaler    = torch.amp.GradScaler('cuda')
        scheduler = optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=Config.epochs, eta_min=1e-6
        )

        best_qwk          = 0.0
        patience          = 4
        epochs_no_improve = 0

        for epoch in range(Config.epochs):
            train_loss = train_one_epoch(
                model, train_loader, criterion, optimizer, scaler, Config.device
            )
            val_loss, qwk, acc = valid_one_epoch(
                model, valid_loader, criterion, Config.device
            )
            scheduler.step()

            print(
                f'Epoch {epoch+1:02d}/{Config.epochs} | '
                f'Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | '
                f'QWK: {qwk:.4f} | Acc: {acc:.4f}'
            )

            if qwk > best_qwk:
                best_qwk          = qwk
                epochs_no_improve = 0
                torch.save(model.state_dict(), ckpt_path)
                print(f'  >>> Saved fold {fold} (QWK: {qwk:.4f})')
            else:
                epochs_no_improve += 1
                print(f'  No improvement ({epochs_no_improve}/{patience})')
                if epochs_no_improve >= patience:
                    print(f'  Early stopping at epoch {epoch+1}')
                    break

        print(f'Fold {fold} best QWK: {best_qwk:.4f}')


# ── HOW TO USE ────────────────────────────────────────────────────────────────
# Train ConvNeXt from scratch:
#   Config.model_name = 'convnext'
#   run_training(start_fold=0)
#
# Train Swin from scratch:
#   Config.model_name = 'swin'
#   run_training(start_fold=0)
#
# Resume ConvNeXt from existing checkpoints:
#   Config.model_name = 'convnext'
#   run_training(start_fold=0, resume_ckpt_dir='/kaggle/input/datasets/adityaghoshk/wallahi')
#
# Start from fold 2:
#   run_training(start_fold=2)
# ─────────────────────────────────────────────────────────────────────────────

run_training(start_fold=0)


[INFO] Checkpoint directory: /kaggle/working
[INFO] Training model: swin
Class alpha weights: [0.0525 0.1569 0.1197 0.3651 0.3058]

[INFO] Training folds 0 to 4

  FOLD 0 / 4
[FRESH] Starting from ImageNet weights


Training:   0%|          | 0/5762 [00:00<?, ?it/s]

Epoch 01/20 | Train Loss: 0.1200 | Val Loss: 0.0607 | QWK: 0.6467 | Acc: 0.3359
  >>> Saved fold 0 (QWK: 0.6467)


Training:   0%|          | 0/5762 [00:00<?, ?it/s]

Epoch 02/20 | Train Loss: 0.0958 | Val Loss: 0.0485 | QWK: 0.7524 | Acc: 0.5386
  >>> Saved fold 0 (QWK: 0.7524)


...

Training:   0%|          | 0/5763 [00:00<?, ?it/s]

Epoch 08/20 | Train Loss: 0.0530 | Val Loss: 0.0289 | QWK: 0.7340 | Acc: 0.5881
  No improvement (1/4)


Training:   0%|          | 0/5763 [00:00<?, ?it/s]

Epoch 09/20 | Train Loss: 0.0499 | Val Loss: 0.0202 | QWK: 0.8069 | Acc: 0.7043
  No improvement (2/4)


Training:   0%|          | 0/5763 [00:00<?, ?it/s]

In [ ]:
# ─── CELL 8: ENSEMBLE EVALUATION ─────────────────────────────────────────────
# Add all checkpoint paths here — mix ConvNeXt and Swin for best results

EVAL_TEST_DIR = os.path.join(BASE_PATH, 'augmented_resized_V2', 'test')
EVAL_IMG_SIZE = 224
EVAL_BATCH    = 32
EVAL_DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Update these paths ────────────────────────────────────────────────────────
checkpoint_paths = [
    # ConvNeXt 384 checkpoints (your best existing)
    '/kaggle/input/datasets/adityaghoshk/wallahi/convnext_fold0_384_best.pth',
    '/kaggle/input/datasets/adityaghoshk/wallahi/convnext_fold1_384_best.pth',
    '/kaggle/input/datasets/adityaghoshk/wallahi/convnext_fold2_384_best.pth',
    '/kaggle/input/datasets/adityaghoshk/wallahi/convnext_fold3_384_best.pth',
    '/kaggle/input/datasets/adityaghoshk/wallahi/convnext_fold4_384_best.pth',
    # Swin checkpoints (add after training)
    # '/kaggle/working/swin_fold0_best.pth',
    # '/kaggle/working/swin_fold1_best.pth',
    # '/kaggle/working/swin_fold2_best.pth',
    # '/kaggle/working/swin_fold3_best.pth',
    # '/kaggle/working/swin_fold4_best.pth',
]

# Model name per checkpoint — must match architecture used during training
checkpoint_models = [
    'convnext', 'convnext', 'convnext', 'convnext', 'convnext',
    # 'swin', 'swin', 'swin', 'swin', 'swin',
]


# Test dataset
class DRTestDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.samples   = []
        self.transform = transform
        for class_name in sorted(os.listdir(root_dir)):
            class_path = os.path.join(root_dir, class_name)
            if not os.path.isdir(class_path): continue
            label = int(class_name)
            for file in os.listdir(class_path):
                if file.lower().endswith(('.png', '.jpg', '.jpeg')):
                    self.samples.append((os.path.join(class_path, file), label))

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = cv2.imread(path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = circular_crop(img)  # apply circular crop
        if self.transform:
            img = self.transform(image=img)['image']
        return img, torch.tensor(label, dtype=torch.long)


test_transform = A.Compose([
    A.Resize(EVAL_IMG_SIZE, EVAL_IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

test_dataset = DRTestDataset(EVAL_TEST_DIR, transform=test_transform)
test_loader  = DataLoader(
    test_dataset, batch_size=EVAL_BATCH,
    shuffle=False, num_workers=NUM_WORKERS
)
print(f'Total Test Samples: {len(test_dataset)}')

# Ensemble inference
ensemble_probs = None
all_labels     = None
folds_used     = 0

for ckpt_path, model_name in zip(checkpoint_paths, checkpoint_models):
    if not os.path.exists(ckpt_path):
        print(f'[SKIP] Not found: {ckpt_path}')
        continue

    print(f'[INFO] Loading {model_name}: {ckpt_path}')
    model = DRModel(model_name, num_classes=5, pretrained=False)
    model.load_state_dict(torch.load(ckpt_path, map_location=EVAL_DEVICE))
    model.to(EVAL_DEVICE)
    model.eval()

    fold_probs, fold_labels = [], []
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(EVAL_DEVICE)
            probs  = torch.softmax(model(images), dim=1).cpu().numpy()
            fold_probs.append(probs)
            fold_labels.append(labels.numpy())

    fold_probs = np.concatenate(fold_probs)
    if ensemble_probs is None:
        ensemble_probs = fold_probs
        all_labels     = np.concatenate(fold_labels)
    else:
        ensemble_probs += fold_probs

    folds_used += 1
    print(f'  Done. ({folds_used} models loaded so far)')

# Final predictions
y_probs = ensemble_probs / folds_used
y_true  = all_labels
y_pred  = np.argmax(y_probs, axis=1)

print(f'\nEnsemble used {folds_used} checkpoints.')

acc   = accuracy_score(y_true, y_pred)
qwk   = cohen_kappa_score(y_true, y_pred, weights='quadratic')
y_bin = label_binarize(y_true, classes=[0,1,2,3,4])
roc   = roc_auc_score(y_bin, y_probs, multi_class='ovr')

print('\n================ TEST RESULTS ================')
print(f'Accuracy              : {acc:.4f}')
print(f'QWK                   : {qwk:.4f}')
print(f'ROC-AUC (macro OVR)   : {roc:.4f}')
print('==============================================\n')
print(classification_report(
    y_true, y_pred,
    target_names=['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative']
))
